In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
data_for_confounding = pd.read_csv("../../Data/3. AnalysisData/data_for_counfounding.csv")

# Check for association with Bechdel score (0-3)

In [ ]:
# Variables
metrics = [
    'cossim',
    'flesch_diff_m_f',
    'ratio_diff',
    'hedge_diff',
    'jsd_m_f',
    'diff_imperatives_normed'
]
controls = ['prop_fem', 'year', 'imdb_score']
IV = 'bechdel_score'

# Prepare data
df = data_for_confounding.copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=metrics + controls + [IV])

# Bootstrap function
def bootstrap_pval(data, metric, IV, controls, n_boot=5000, random_state=42):
    rng = np.random.default_rng(random_state)

    # Original model
    X = data[[IV] + controls]
    X = sm.add_constant(X)
    y = data[metric]
    model = sm.OLS(y, X).fit()
    orig_coef = model.params[IV]

    # Bootstrap resampling
    boot_coefs = []
    for _ in range(n_boot):
        sample_idx = rng.choice(len(data), size=len(data), replace=True)
        sample = data.iloc[sample_idx]
        Xb = sample[[IV] + controls]
        Xb = sm.add_constant(Xb)
        yb = sample[metric]
        try:
            boot_model = sm.OLS(yb, Xb).fit()
            boot_coefs.append(boot_model.params[IV])
        except:
            continue

    boot_coefs = np.array(boot_coefs)

    # Two-sided empirical p-value
    pval = np.mean(np.abs(boot_coefs) >= np.abs(orig_coef))

    return orig_coef, pval, model

# Run for all metrics
results = []
for metric in metrics:
    coef, pval, model = bootstrap_pval(df, metric, IV, controls)
    # Find strongest control by absolute coef
    control_coefs = model.params[controls]
    strongest_control = control_coefs.abs().idxmax()
    results.append({
        'metric': metric,
        'coef': coef,
        'pval': pval,
        'significant': pval < 0.05,
        'strongest_control': strongest_control,
        'strongest_control_coef': model.params[strongest_control],
        'strongest_control_pval': model.pvalues[strongest_control]
    })

# Summary
for res in results:
    sig_str = "SIGNIFICANT" if res['significant'] else "not significant"
    print(f"{res['metric']}: {sig_str} (coef={res['coef']:.3f}, p={res['pval']:.3f}) | "
          f"Strongest control: {res['strongest_control']} "
          f"(coef={res['strongest_control_coef']:.3f}, p={res['strongest_control_pval']:.3f})")


cossim: not significant (coef=0.019, p=0.456) | Strongest control: prop_fem (coef=0.062, p=0.022)
flesch_diff_m_f: not significant (coef=-0.570, p=0.497) | Strongest control: prop_fem (coef=-4.846, p=0.066)
ratio_diff: not significant (coef=0.015, p=0.504) | Strongest control: prop_fem (coef=-0.386, p=0.000)
hedge_diff: not significant (coef=0.751, p=0.505) | Strongest control: imdb_score (coef=1.121, p=0.009)
jsd_m_f: not significant (coef=-0.017, p=0.472) | Strongest control: prop_fem (coef=-0.033, p=0.103)
diff_imperatives_normed: not significant (coef=0.002, p=0.560) | Strongest control: prop_fem (coef=-0.027, p=0.017)


- ols with bootstrapping
- none significant after controlling, prop_fem a strong predictor for most of them
- but seems to contradict what upcoming results show?

# Check for association with Bechdel passing (binary pass/fail)

In [ ]:
import statsmodels.api as sm
import numpy as np

metrics = [
    'cossim',
    'flesch_diff_m_f',
    'ratio_diff',
    'hedge_diff',
    'jsd_m_f',
    'diff_imperatives_normed'
]

controls = ['prop_fem', 'year', 'imdb_score']
IV = 'bechdel_score_hard'  

df = data_for_confounding.copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=metrics + controls + [IV])

results_summary = []

for metric in metrics:
    X = df[[IV] + controls]
    X = sm.add_constant(X)
    y = df[metric]
    
    model = sm.OLS(y, X).fit(cov_type='HC3') 
    
    coef = model.params[IV]
    pval = model.pvalues[IV]
    
    control_coefs = model.params[controls]
    strongest_control = control_coefs.abs().idxmax()
    
    results_summary.append({
        'metric': metric,
        'coef': coef,
        'pval': pval,
        'significant': pval < 0.05,
        'strongest_control': strongest_control,
        'strongest_control_coef': model.params[strongest_control],
        'strongest_control_pval': model.pvalues[strongest_control]
    })

summary_df = pd.DataFrame(results_summary)

for _, row in summary_df.iterrows():
    sig_str = "SIGNIFICANT" if row['significant'] else "not significant"
    print(f"{row['metric']}: {sig_str} (coef={row['coef']:.3f}, p={row['pval']:.3f}) | "
          f"Strongest control: {row['strongest_control']} "
          f"(coef={row['strongest_control_coef']:.3f}, p={row['strongest_control_pval']:.3f})")


cossim: SIGNIFICANT (coef=0.023, p=0.002) | Strongest control: prop_fem (coef=0.087, p=0.077)
flesch_diff_m_f: not significant (coef=-0.657, p=0.342) | Strongest control: prop_fem (coef=-5.595, p=0.051)
ratio_diff: SIGNIFICANT (coef=0.026, p=0.018) | Strongest control: prop_fem (coef=-0.378, p=0.000)
hedge_diff: SIGNIFICANT (coef=2.605, p=0.005) | Strongest control: imdb_score (coef=1.146, p=0.007)
jsd_m_f: SIGNIFICANT (coef=-0.023, p=0.000) | Strongest control: prop_fem (coef=-0.051, p=0.132)
diff_imperatives_normed: not significant (coef=0.001, p=0.718) | Strongest control: prop_fem (coef=-0.025, p=0.063)


- all significant except FRES and imperatives which were never significant in the first place (w/o controlling)...so even though prop_fem was often a good control there's still an association by pass/fail?


- not sure about GLM/if it fits? what distribution to use?